In [ ]:
# 1. Create a new workspace user (innowise@innowise.com) and add them to a new group named
# innowise.

In [2]:
import os
from src.core.services.connection.databricks_connector import DatabricksConnectionService

conn = DatabricksConnectionService(app_name="testing-pipeline")
conn.connect()
spark = conn.spark


2026-10-08 10:51:00.159 | INFO     | src.core.services.connection.databricks_connector:connect:34 - SparkSession ready | app=testing-pipeline | version=4.2.0 | runtime=serverless-connect


In [3]:
# 1. Даем доступ к каталогу
spark.sql("GRANT USE CATALOG ON CATALOG yandex_go_dev TO `innowise`")

# 2. Даем доступ к схеме Gold
spark.sql("GRANT USE SCHEMA ON SCHEMA yandex_go_dev.gold TO `innowise`")

# 3. Даем право чтения таблиц схемы Gold
spark.sql("GRANT SELECT ON SCHEMA yandex_go_dev.gold TO `innowise`")

# 4. Проверяем выданные права (выводим результат)
grants_df = spark.sql("SHOW GRANTS ON SCHEMA yandex_go_dev.gold")
display(grants_df)

DataFrame[Principal: string, ActionType: string, ObjectType: string, ObjectKey: string]

In [4]:
# 5. Отзываем право SELECT:
spark.sql("REVOKE SELECT ON SCHEMA yandex_go_dev.gold FROM `innowise`")

# 6. Проверяем, что SELECT пропал:
display(spark.sql("SHOW GRANTS ON SCHEMA yandex_go_dev.gold"))

DataFrame[Principal: string, ActionType: string, ObjectType: string, ObjectKey: string]

In [5]:
import json
from src.core.services.audit.reporting import WorkflowReportService

reporter = WorkflowReportService()
report = reporter.generate_report(spark)

print(json.dumps(report, indent=2))

2026-10-08 10:52:05.289 | INFO     | src.core.services.audit.reporting:generate_report:54 - Generating Lakehouse Audit Report for catalog [yandex_go_dev]...
2026-10-08 10:52:59.237 | INFO     | src.core.services.audit.reporting:generate_report:88 - Audit report saved to: /Volumes/yandex_go_dev/raw_files/landing/reports/workflow_summary.json


{
  "pipeline_name": "yandex-go-analytics",
  "environment": "dev",
  "catalog": "yandex_go_dev",
  "generated_at": "2026-10-08T07:52:55.610041Z",
  "summary": {
    "total_monitored_tables": 10,
    "total_rows_across_lakehouse": 35415,
    "pipeline_status": "SUCCESS"
  },
  "layers": {
    "bronze": {
      "taxi": {
        "exists": true,
        "row_count": 10000,
        "latest_version": 4,
        "latest_operation": "STREAMING UPDATE",
        "last_modified": "2026-10-05 11:25:50",
        "status": "HEALTHY"
      },
      "drivers": {
        "exists": true,
        "row_count": 250,
        "latest_version": 4,
        "latest_operation": "STREAMING UPDATE",
        "last_modified": "2026-10-03 18:03:19",
        "status": "HEALTHY"
      },
      "users": {
        "exists": true,
        "row_count": 1500,
        "latest_version": 2,
        "latest_operation": "STREAMING UPDATE",
        "last_modified": "2026-10-05 11:25:40",
        "status": "HEALTHY"
      }
    